# Part of speech 
This page examines the performance of DaCy and competing Danish models in the task of part-of-speech tagging. The general accuracy is evaluated on two datasets, dacy data and hist-pos-dk. Accuracy is reported with 95% confidence intervals calculated using bootstrapping with 100 samples. Apart from accuracy, the evaluation includes the models' robustness and checks for biases.

# Historical data
To compare the models performance on historical data, we used the [hist-pos-dk](https://huggingface.co/datasets/chcaa/hist-dk-pos) dataset. It contains example sentences extracted from a piece of historical Danish newspaper fiction published in 1824, which include archaic Danish spelling (eg....). 

The table below shows the pos accuracy of Danish language models on the testing data, that newer dacy models were trained on, and on the historical dataset. The best results according to a model and a dataset are highlited by a bold font, the second best are underlined.

In [ ]:
import random

import numpy as np
import pandas as pd
import sys; sys.path.insert(0, "/work/JohanaMayerova#0832/DaCy/docs")
from spacy.scorer import Scorer
from spacy.tokens import Doc
from spacy.training import Example
from evaluation.models import MODELS_POS
from evaluation.datasets import datasets
from evaluation.utils import bootstrap_pos_ci

# load datasets
dacy_data = datasets.get("dacy_data")()   
hist_pos = datasets.get("hist_pos_dk")() 


def evaluate_pos(nlp, gold_examples: list) -> dict:
    # run model on pre-segmented tokens
    word_docs = [Doc(nlp.vocab, words=[t.text for t in ex.reference]) for ex in gold_examples]
    pred_docs = list(nlp.pipe(word_docs, batch_size=32))
    examples = [Example(pred, ex.reference) for pred, ex in zip(pred_docs, gold_examples)]
    # keep per-example (n_correct, n_tokens) so we can bootstrap CIs later
    per_example = [
        (
            sum(p.pos_ == g.pos_ for p, g in zip(ex.predicted, ex.reference)),
            len(ex.reference),
        )
        for ex in examples
    ]
    return {
        "scores": Scorer.score_token_attr(examples, "pos"),
        "per_example": per_example,
    }


# evaluate every model on both datasets
results = {}
for mdl_name, model_getter in MODELS_POS.items():
    nlp = model_getter()
    results[mdl_name] = {
        "dacy_data": evaluate_pos(nlp, dacy_data["test"]),
        "hist_pos_dk": evaluate_pos(nlp, hist_pos["train"]),
    }


Token indices sequence length is longer than the specified maximum sequence length for this model (162 > 128). Running this sequence through the model will result in indexing errors
/work/JohanaMayerova#0832/dacy_pos/dacy_eval/lib/python3.12/site-packages/spacy/pipeline/attributeruler.py:138: UserWarning: [W036] The component 'matcher' does not have any patterns defined.
  matches = self.matcher(doc, allow_missing=True, as_spans=False)


In [8]:
def parse_score(value) -> float:
    """Extract the point estimate from 'acc (lo, hi)' CI strings (or pass floats through)."""
    if isinstance(value, str):
        return float(value.split()[0])
    return float(value)


def highlight_max(s: pd.Series) -> list:
    """Highlight the maximum in a Series with bold text."""
    s = s.map(parse_score)
    is_max = s == s.max()
    return ["font-weight: bold" if v else "" for v in is_max]


def underline_second_max(s: pd.Series) -> list:
    """Underline the second maximum in a Series."""
    s = s.map(parse_score)
    is_second_max = s == s.sort_values(ascending=False).iloc[1]
    return ["text-decoration: underline" if v else "" for v in is_second_max]


def create_pos_table(
    df: pd.DataFrame,
    caption="POS accuracy. The best score is highlighted in bold and the second best is underlined.",
):
    s = df.style.apply(highlight_max, axis=0, subset=df.columns[1:])
    if len(df) > 1:
        s = s.apply(underline_second_max, axis=0, subset=df.columns[1:])
    s = s.set_caption(caption)
    s = s.set_properties(subset=df.columns[1:], **{"text-align": "right"})
    s = s.set_properties(subset=[df.columns[0]], **{"text-align": "left"})
    s = s.hide(axis="index")
    s = s.set_table_attributes('style="font-size: 0.65em"')
    return s

In [9]:
rows = []
for mdl_name, res in results.items():
    row = {"Model": mdl_name}
    for ds, key in [("dacy_data", "dacy_data"), ("hist-pos-dk", "hist_pos_dk")]:
        acc = 100 * res[key]["scores"]["pos_acc"]
        ci = 100 * bootstrap_pos_ci(res[key]["per_example"])
        row[ds] = f"{acc:.2f} ({ci[0]:.2f}, {ci[1]:.2f})"
    rows.append(row)
df_pos = pd.DataFrame(rows)
create_pos_table(
    df_pos,
    "POS tagging accuracy (%) on the dacy-data test set and hist-pos-dk. "
    "95% confidence intervals calculated using bootstrapping with 100 samples.",
)

Model,dacy_data,hist-pos-dk
da_dacy_small_trf-1.0.0,"97.81 (97.39, 98.27)","94.32 (92.56, 95.75)"
da_dacy_large_trf-1.0.0,"98.98 (98.70, 99.20)","95.60 (93.65, 97.11)"
da_dacy_edge-0.1.0,"94.72 (94.23, 95.31)","87.50 (85.18, 89.73)"
